# Animation Format

by Jerome Eippers, 2024

## Skeleton structure 
A character is composed of a hiearchy of joint. Each with a local transfrom relative to its parent.
       
```
Root
└── Hips
    ├── Spine
    │   └── Chest
    │       ├── Neck
    │       │   └── Head
    │       ├── LeftShoulder
    │       │   └── LeftArm
    │       │       └── (...)
    │       └── RightShoulder
    │           └── RightArm
    │               └── (...)
    ├── LeftUpLeg
    │   └── LeftLeg
    │       └── LeftFoot
    └── RightUpLeg
        └── RightLeg
            └── RightFoot
```

## Animation representation

$$
M(t) = (\textbf{p}_0(t), \textbf{q}_0(t), \textbf{q}_1(t), \textbf{q}_2(t), \dots, \textbf{q}_b(t))
$$
  
An articulated objects such as human figures are usually represented
as rotation hierarchies parameterized by a whole-body translation $\textbf{p}_0$, a
whole-body rotation $\textbf{q}_0$, and a set of joint angles $\textbf{q}_1, \dots, \textbf{q}_b$.   
A motion $M$ is described by a set of motion curves each giving the value of one of the model’s parameters as a function of time.

### Framework

In our case, we use the animation format from Ubisoft Lafan1 with a motion being parametrized by the position and orientation of all joints.

$$
\begin{align*}
M(t) &= (\textbf{P}(t), \textbf{Q}(t))\\
\textbf{P}(t) &= (\textbf{p}_0(t), \dots, \textbf{p}_b(t))\\
\textbf{Q}(t) &= (\textbf{q}_0(t), \dots, \textbf{q}_b(t))\\
\end{align*}
$$
With $t$ beeing discreet at 30 frame per second.  

This gives us 2 tensors of shapes :

* $\textbf{P}$ = np.array[frame count, bone count, 3]
* $\textbf{Q}$ = np.array[frame count, bone count, 4]


In [1]:
import numpy as np
from ipywidgets import widgets, interact

import ipyanimlab as lab

viewer = lab.Viewer(move_speed=5, width=1280, height=720)

# Load Character

Mesh and Skeleton.

In [2]:
character = viewer.import_usd_asset('AnimLabSimpleMale.usd')

In [3]:
viewer.begin_shadow()
viewer.draw(character)
viewer.end_shadow()

viewer.begin_display()
viewer.draw_ground()
viewer.draw(character)
viewer.end_display()

viewer.disable(depth_test=True)
viewer.draw_axis(character.world_skeleton_xforms(), 5)
viewer.draw_lines(character.world_skeleton_lines())
viewer.execute_commands()
viewer

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

# Load Animation

## load the data

In [4]:
animation = lab.import_bvh('../../resources/lafan1/bvh/aiming1_subject1.bvh')

In [6]:
print('positions :', animation.pos.shape)
print('quaternions :', animation.quats.shape)

positions : (7184, 22, 3)
quaternions : (7184, 22, 4)


In [6]:
np.set_printoptions(precision=4, suppress=True)

print("positions shape:", animation.pos.shape)
print("quaternions shape:", animation.quats.shape)

print("\n第0帧所有骨骼的位置：")
print(animation.pos[0])

print("\n第0帧所有骨骼的四元数：")
print(animation.quats[0])

print("\n前2帧、前3根骨骼的位置：")
print(animation.pos[:2, :3, :])

print("\n前2帧、前3根骨骼的四元数：")
print(animation.quats[:2, :3, :])

positions shape: (7184, 22, 3)
quaternions shape: (7184, 22, 4)

第0帧所有骨骼的位置：
[[-224.6895   91.8821 -431.6255]
 [   0.1035    1.8578   10.5485]
 [  43.5      -0.        0.    ]
 [  42.3722    0.        0.    ]
 [  17.3      -0.        0.    ]
 [   0.1035    1.8578  -10.5485]
 [  43.5      -0.        0.    ]
 [  42.3723    0.        0.    ]
 [  17.3      -0.        0.    ]
 [   6.902    -2.6037   -0.    ]
 [  12.5881    0.        0.    ]
 [  12.3432   -0.       -0.    ]
 [  25.8329    0.        0.    ]
 [  11.7666   -0.       -0.    ]
 [  19.7459   -1.4803    6.0001]
 [  11.2841    0.       -0.    ]
 [  33.       -0.        0.    ]
 [  25.2       0.        0.    ]
 [  19.7461   -1.4804   -6.0001]
 [  11.2841   -0.       -0.    ]
 [  33.0001    0.       -0.    ]
 [  25.1998    0.0001    0.0004]]

第0帧所有骨骼的四元数：
[[ 0.5212  0.4602  0.5277  0.488 ]
 [ 0.0129  0.0536 -0.9983 -0.0185]
 [ 0.9946 -0.0742 -0.007  -0.0721]
 [ 0.8056 -0.0595  0.0053  0.5895]
 [ 0.9825 -0.      0.      0.1861]
 [ 0.06

In [5]:
def render(frame):
    
    p = (animation.pos[frame,...]).copy()
    q = (animation.quats[frame,...]).copy()

    a = lab.utils.quat_to_mat(q, p)
    viewer.set_shadow_poi(a[1,:3,3])
    
    viewer.begin_shadow()
    viewer.draw(character, a, animation.bones)
    viewer.end_shadow()
    
    viewer.begin_display()
    viewer.draw_ground()
    viewer.draw(character, a, animation.bones)
    viewer.end_display()
    
    viewer.disable(depth_test=True)
        
    viewer.draw_axis(character.world_skeleton_xforms(a, animation.bones), 5)
    viewer.draw_lines(character.world_skeleton_lines(a, animation.bones))
    
    viewer.execute_commands()
    
interact(render, frame=lab.Timeline(max=animation.quats.shape[0]-1))
viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

In [8]:
def render_skeleton(frame):
    p = (animation.pos[frame, ...]).copy()
    q = (animation.quats[frame, ...]).copy()

    a = lab.utils.quat_to_mat(q, p)

    # 不再画角色 mesh，也不再需要 shadow pass
    viewer.begin_display()
    viewer.draw_ground()   # 想保留地面就留着；不要地面可删掉
    viewer.end_display()

    # 只画骨架
    viewer.disable(depth_test=True)
    viewer.draw_axis(character.world_skeleton_xforms(a, animation.bones), 5)
    viewer.draw_lines(character.world_skeleton_lines(a, animation.bones))

    viewer.execute_commands()

interact(render_skeleton, frame=lab.Timeline(max=animation.quats.shape[0]-1))
viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

## Match the skeleton translations

In [4]:
animmap = lab.AnimMapper(character)
animation = lab.import_bvh('../../resources/lafan1/bvh/aiming1_subject1.bvh', anim_mapper=animmap)

In [5]:
print("animmap =", animmap)
print("type(animmap) =", type(animmap))
print("dir(animmap) =")
print(dir(animmap))

animmap = <ipyanimlab.animation.AnimMapper object at 0x0000027AE5279780>
type(animmap) = <class 'ipyanimlab.animation.AnimMapper'>
dir(animmap) =
['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'asset', 'effector_names', 'effector_offests', 'keep_translation', 'local_offsets', 'match_effectors', 'mirror', 'mirror_left', 'mirror_right', 'root_motion']


In [6]:
def render(frame):
    
    p = (animation.pos[frame,...]).copy()
    q = (animation.quats[frame,...]).copy()

    a = lab.utils.quat_to_mat(q, p)
    viewer.set_shadow_poi(a[1,:3,3])
    
    viewer.begin_shadow()
    viewer.draw(character, a, animation.bones)
    viewer.end_shadow()
    
    viewer.begin_display()
    viewer.draw_ground()
    viewer.draw(character, a, animation.bones)
    viewer.end_display()

    viewer.disable(depth_test=True)
        
    viewer.draw_axis(character.world_skeleton_xforms(a), 5)
    viewer.draw_lines(character.world_skeleton_lines(a))
    
    viewer.execute_commands()
    
interact(render, frame=lab.Timeline(max=animation.quats.shape[0]-1))
viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

## Comparsion

In [7]:
character_raw = viewer.import_usd_asset('AnimLabSimpleMale.usd')
character_mapped = viewer.import_usd_asset('AnimLabSimpleMale.usd')

animation_raw = lab.import_bvh('../../resources/lafan1/bvh/aiming1_subject1.bvh')

animmap = lab.AnimMapper(character_mapped)
animation_mapped = lab.import_bvh(
    '../../resources/lafan1/bvh/aiming1_subject1.bvh',
    anim_mapper=animmap
)

In [8]:
def render_compare(frame, offset=60, show_skeleton=False):
    # -------- 左边：不带 map --------
    p_raw = animation_raw.pos[frame, ...].copy()
    q_raw = animation_raw.quats[frame, ...].copy()

    # 给左边角色一个负的 x 偏移
    p_raw[0, 0] -= offset

    a_raw = lab.utils.quat_to_mat(q_raw, p_raw)

    # -------- 右边：带 map --------
    p_map = animation_mapped.pos[frame, ...].copy()
    q_map = animation_mapped.quats[frame, ...].copy()

    # 给右边角色一个正的 x 偏移
    p_map[0, 0] += offset

    a_map = lab.utils.quat_to_mat(q_map, p_map)

    # -------- shadow pass --------
    viewer.begin_shadow()
    viewer.draw(character_raw, a_raw, animation_raw.bones)
    viewer.draw(character_mapped, a_map)
    viewer.end_shadow()

    # -------- display pass --------
    viewer.begin_display()
    viewer.draw_ground()
    viewer.draw(character_raw, a_raw, animation_raw.bones)
    viewer.draw(character_mapped, a_map)
    viewer.end_display()

    # -------- 可选：叠加骨架线 --------
    if show_skeleton:
        viewer.disable(depth_test=True)
        viewer.draw_axis(character_raw.world_skeleton_xforms(a_raw, animation_raw.bones), 4)
        viewer.draw_lines(character_raw.world_skeleton_lines(a_raw, animation_raw.bones))

        viewer.draw_axis(character_mapped.world_skeleton_xforms(a_map), 4)
        viewer.draw_lines(character_mapped.world_skeleton_lines(a_map))

    viewer.execute_commands()


interact(
    render_compare,
    frame=lab.Timeline(max=animation_raw.quats.shape[0] - 1),
    offset=(20, 120, 5),
    show_skeleton=False
)

viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

In [10]:
anim_raw = lab.import_bvh('../../resources/lafan1/bvh/aiming1_subject1.bvh')

animmap = lab.AnimMapper(character)
anim_mapped = lab.import_bvh(
    '../../resources/lafan1/bvh/aiming1_subject1.bvh',
    anim_mapper=animmap
)

np.set_printoptions(precision=4, suppress=True)

def inspect_animation(name, anim, frame=0, bone_count=22):
    print(f"===== {name} =====")
    print("pos.shape:", anim.pos.shape)
    print("quats.shape:", anim.quats.shape)

    if hasattr(anim, "bones"):
        print("first bones:", anim.bones[:bone_count])

    if hasattr(anim, "parents"):
        print("first parents:", anim.parents[:bone_count])

    print(f"\nframe {frame} first {bone_count} bones positions:")
    print(anim.pos[frame, :bone_count, :])

    print(f"\nframe {frame} first {bone_count} bones quaternions:")
    print(anim.quats[frame, :bone_count, :])

    print("\n")

inspect_animation("RAW BVH", anim_raw)
inspect_animation("MAPPED TO CHARACTER", anim_mapped)

print("===== position diff (mapped - raw), frame 0, first 5 bones =====")
print(anim_mapped.pos[0, :22, :] - anim_raw.pos[0, :22, :])

print("\n===== quaternion diff (mapped - raw), frame 0, first 5 bones =====")
print(anim_mapped.quats[0, :22, :] - anim_raw.quats[0, :22, :])

===== RAW BVH =====
pos.shape: (7184, 22, 3)
quats.shape: (7184, 22, 4)
first bones: ['Hips', 'LeftUpLeg', 'LeftLeg', 'LeftFoot', 'LeftToe', 'RightUpLeg', 'RightLeg', 'RightFoot', 'RightToe', 'Spine', 'Spine1', 'Spine2', 'Neck', 'Head', 'LeftShoulder', 'LeftArm', 'LeftForeArm', 'LeftHand', 'RightShoulder', 'RightArm', 'RightForeArm', 'RightHand']
first parents: [-1  0  1  2  3  0  5  6  7  0  9 10 11 12 11 14 15 16 11 18 19 20]

frame 0 first 22 bones positions:
[[-224.6895   91.8821 -431.6255]
 [   0.1035    1.8578   10.5485]
 [  43.5      -0.        0.    ]
 [  42.3722    0.        0.    ]
 [  17.3      -0.        0.    ]
 [   0.1035    1.8578  -10.5485]
 [  43.5      -0.        0.    ]
 [  42.3723    0.        0.    ]
 [  17.3      -0.        0.    ]
 [   6.902    -2.6037   -0.    ]
 [  12.5881    0.        0.    ]
 [  12.3432   -0.       -0.    ]
 [  25.8329    0.        0.    ]
 [  11.7666   -0.       -0.    ]
 [  19.7459   -1.4803    6.0001]
 [  11.2841    0.       -0.    ]
 [  3

In [11]:
def print_skeleton_tree(bones, parents, title="Skeleton"):
    print(f"\n===== {title} TREE =====")
    
    children = {i: [] for i in range(len(bones))}
    roots = []

    for i, p in enumerate(parents):
        if p == -1:
            roots.append(i)
        else:
            children[p].append(i)

    def dfs(node, indent=""):
        print(f"{indent}- [{node}] {bones[node]}")
        for child in children[node]:
            dfs(child, indent + "  ")

    for r in roots:
        dfs(r)


def print_bone_parent_table(bones, parents, title="Skeleton"):
    print(f"\n===== {title} BONE -> PARENT =====")
    for i, (b, p) in enumerate(zip(bones, parents)):
        parent_name = "None" if p == -1 else bones[p]
        print(f"[{i:02d}] {b:<15} parent = [{p:02d}] {parent_name}")


print_skeleton_tree(anim_raw.bones, anim_raw.parents, "RAW BVH")
print_bone_parent_table(anim_raw.bones, anim_raw.parents, "RAW BVH")

print_skeleton_tree(anim_mapped.bones, anim_mapped.parents, "MAPPED TO CHARACTER")
print_bone_parent_table(anim_mapped.bones, anim_mapped.parents, "MAPPED TO CHARACTER")


===== RAW BVH TREE =====
- [0] Hips
  - [1] LeftUpLeg
    - [2] LeftLeg
      - [3] LeftFoot
        - [4] LeftToe
  - [5] RightUpLeg
    - [6] RightLeg
      - [7] RightFoot
        - [8] RightToe
  - [9] Spine
    - [10] Spine1
      - [11] Spine2
        - [12] Neck
          - [13] Head
        - [14] LeftShoulder
          - [15] LeftArm
            - [16] LeftForeArm
              - [17] LeftHand
        - [18] RightShoulder
          - [19] RightArm
            - [20] RightForeArm
              - [21] RightHand

===== RAW BVH BONE -> PARENT =====
[00] Hips            parent = [-1] None
[01] LeftUpLeg       parent = [00] Hips
[02] LeftLeg         parent = [01] LeftUpLeg
[03] LeftFoot        parent = [02] LeftLeg
[04] LeftToe         parent = [03] LeftFoot
[05] RightUpLeg      parent = [00] Hips
[06] RightLeg        parent = [05] RightUpLeg
[07] RightFoot       parent = [06] RightLeg
[08] RightToe        parent = [07] RightFoot
[09] Spine           parent = [00] Hips
[10] Spine1 

## Project Root

We will project the root under the Hips.

In [8]:
frame_count = animation.quats.shape[0]
bone_count = animation.quats.shape[1]

hips_v = lab.utils.quat_mul_vec(animation.quats[:, 1], np.asarray([0,1,0], dtype=np.float32)[np.newaxis,...].repeat(frame_count, axis=0))
angle = np.arctan2(hips_v[:, 0], hips_v[:, 2])/2
root_q = np.zeros([frame_count, 4], dtype=np.float32)
root_q[:, 0] = np.cos(angle)
root_q[:, 2] = np.sin(angle)
root_p = np.zeros([frame_count, 3], dtype=np.float32)
root_p[:, [0,2]] = animation.pos[:, 1, [0,2]]

animation.quats[:, 1], animation.pos[:, 1] = lab.utils.qp_mul( lab.utils.qp_inv((root_q, root_p)), (animation.quats[:, 1], animation.pos[:, 1]) )
animation.quats[:, 0], animation.pos[:, 0] = root_q, root_p

In [9]:
def render(frame, static_position=False, static_rotation=False):
    
    p = (animation.pos[frame,...]).copy()
    q = (animation.quats[frame,...]).copy()
    
    if static_position:
        p[0] = np.zeros(3, dtype=np.float32)
    if static_rotation:
        q[0] = np.asarray([1,0,0,0], dtype=np.float32)
    
    a = lab.utils.quat_to_mat(q, p)
    viewer.set_shadow_poi(a[1,:3,3])
    
    viewer.begin_shadow()
    viewer.draw(character, a, animation.bones)
    viewer.end_shadow()
    
    viewer.begin_display()
    viewer.draw_ground()
    viewer.draw(character, a, animation.bones)
    viewer.end_display()
    
    viewer.disable(depth_test=True)
        
    viewer.draw_axis(character.world_skeleton_xforms(a), 5)
    viewer.draw_lines(character.world_skeleton_lines(a))
    
    viewer.execute_commands()
    
interact(render, 
         static_position=widgets.Checkbox(), 
         static_rotation=widgets.Checkbox(), 
         frame=lab.Timeline(max=animation.quats.shape[0]-1))
viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-18.0, camera_pos=[-370, 280, 350], camera_yaw=-45.0,…

## Project Root Comp

In [7]:
# --- 加载两个角色（左右对比用） ---
character_noinv = viewer.import_usd_asset('AnimLabSimpleMale.usd')
character_inv   = viewer.import_usd_asset('AnimLabSimpleMale.usd')

# 这里假设你前面已经有 animation
# 比如它来自：
# animmap = lab.AnimMapper(character_inv)
# animation = lab.import_bvh('../../resources/lafan1/bvh/aiming1_subject1.bvh', anim_mapper=animmap)

frame_count = animation.quats.shape[0]
bone_count = animation.quats.shape[1]

# 原动画复制两份
rot_noinv = animation.quats.copy()
pos_noinv = animation.pos.copy()

rot_inv = animation.quats.copy()
pos_inv = animation.pos.copy()

# -----------------------------
# 1) 从 hips 提取地面 root
# -----------------------------
hips_v = lab.utils.quat_mul_vec(animation.quats[:, 1], [0, 1, 0])
angle = np.arctan2(hips_v[:, 0], hips_v[:, 2]) / 2.0

root_q = np.zeros((frame_count, 4), dtype=np.float32)
root_q[:, 0] = np.cos(angle)
root_q[:, 2] = np.sin(angle)
root_q = lab.utils.quat_normalize(root_q)

root_p = np.zeros((frame_count, 3), dtype=np.float32)
root_p[:, [0, 2]] = animation.pos[:, 1, [0, 2]]

# -----------------------------
# 2) 错误版：不执行 inverse
#    只把 root 塞到 bone 0
# -----------------------------
rot_noinv[:, 0] = root_q
pos_noinv[:, 0] = root_p

animation_noinv = lab.Anim(
    rot_noinv,
    pos_noinv,
    pos_noinv[0],
    animation.parents,
    animation.bones
)

# -----------------------------
# 3) 正常版：执行 inverse
#    把 root 从 hips 里扣掉
# -----------------------------
rot_inv[:, 1], pos_inv[:, 1] = lab.utils.qp_mul(
    lab.utils.qp_inv((root_q, root_p)),
    (rot_inv[:, 1], pos_inv[:, 1])
)

rot_inv[:, 0] = root_q
pos_inv[:, 0] = root_p

animation_inv = lab.Anim(
    rot_inv,
    pos_inv,
    pos_inv[0],
    animation.parents,
    animation.bones
)

print("animation_noinv 已生成：不执行 inverse 的错误版")
print("animation_inv 已生成：正常 project root 版本")

animation_noinv 已生成：不执行 inverse 的错误版
animation_inv 已生成：正常 project root 版本


In [8]:
def render_project_root_compare(frame, offset=60, show_skeleton=False):
    # -------------------------
    # 左边：不做 inverse 的错误版
    # -------------------------
    p_noinv = animation_noinv.pos[frame, ...].copy()
    q_noinv = animation_noinv.quats[frame, ...].copy()

    # 左移
    p_noinv[0, 0] -= offset

    a_noinv = lab.utils.quat_to_mat(q_noinv, p_noinv)

    # -------------------------
    # 右边：正常 project root 版
    # -------------------------
    p_inv = animation_inv.pos[frame, ...].copy()
    q_inv = animation_inv.quats[frame, ...].copy()

    # 右移
    p_inv[0, 0] += offset

    a_inv = lab.utils.quat_to_mat(q_inv, p_inv)

    # -------------------------
    # shadow pass
    # -------------------------
    viewer.begin_shadow()
    viewer.draw(character_noinv, a_noinv)
    viewer.draw(character_inv, a_inv)
    viewer.end_shadow()

    # -------------------------
    # display pass
    # -------------------------
    viewer.begin_display()
    viewer.draw_ground()
    viewer.draw(character_noinv, a_noinv)
    viewer.draw(character_inv, a_inv)
    viewer.end_display()

    # -------------------------
    # 可选：叠加骨架
    # -------------------------
    if show_skeleton:
        viewer.disable(depth_test=True)

        viewer.draw_axis(character_noinv.world_skeleton_xforms(a_noinv), 4)
        viewer.draw_lines(character_noinv.world_skeleton_lines(a_noinv))

        viewer.draw_axis(character_inv.world_skeleton_xforms(a_inv), 4)
        viewer.draw_lines(character_inv.world_skeleton_lines(a_inv))

    viewer.execute_commands()


interact(
    render_project_root_compare,
    frame=lab.Timeline(max=animation_inv.quats.shape[0] - 1),
    offset=(20, 120, 5),
    show_skeleton=False
)

viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-13.00000000000001, camera_pos=[-210.80523165162083, …

## Project Root Comp 2

In [9]:
def extract_root_hips_world_traj(character, animation, names=None):
    frame_count = animation.quats.shape[0]

    root_world = np.zeros((frame_count, 3), dtype=np.float32)
    hips_world = np.zeros((frame_count, 3), dtype=np.float32)

    for frame in range(frame_count):
        a = lab.utils.quat_to_mat(animation.quats[frame], animation.pos[frame])

        # 如果 names 不为 None，就按骨骼名字做映射
        if names is None:
            w = character.world_skeleton_xforms(a)
        else:
            w = character.world_skeleton_xforms(a, names)

        root_world[frame] = w[0, :3, 3]
        hips_world[frame] = w[1, :3, 3]

    return root_world, hips_world


def polyline_to_segments(points):
    """
    把折线点列:
        [p0, p1, p2, p3]
    变成 draw_lines 需要的线段点列:
        [p0, p1, p1, p2, p2, p3]
    """
    if points.shape[0] < 2:
        return np.zeros((0, 3), dtype=np.float32)

    segs = np.zeros((2 * (points.shape[0] - 1), 3), dtype=np.float32)
    segs[0::2] = points[:-1]
    segs[1::2] = points[1:]
    return segs


# 预计算两套动画的 root / hips 世界轨迹
root_noinv_world, hips_noinv_world = extract_root_hips_world_traj(
    character_noinv, animation_noinv, animation_noinv.bones
)

root_inv_world, hips_inv_world = extract_root_hips_world_traj(
    character_inv, animation_inv, animation_inv.bones
)

print("轨迹预计算完成：")
print("  左边（no inverse） root / hips")
print("  右边（with inverse） root / hips")

轨迹预计算完成：
  左边（no inverse） root / hips
  右边（with inverse） root / hips


In [10]:
def render_project_root_compare_with_traj(
    frame,
    offset=60,
    traj_step=6,
    show_mesh=True,
    show_skeleton=False,
    show_root_traj=True,
    show_hips_traj=True
):
    # -------------------------
    # 左边：不做 inverse
    # -------------------------
    p_noinv = animation_noinv.pos[frame].copy()
    q_noinv = animation_noinv.quats[frame].copy()
    p_noinv[0, 0] -= offset
    a_noinv = lab.utils.quat_to_mat(q_noinv, p_noinv)

    # -------------------------
    # 右边：正常 inverse
    # -------------------------
    p_inv = animation_inv.pos[frame].copy()
    q_inv = animation_inv.quats[frame].copy()
    p_inv[0, 0] += offset
    a_inv = lab.utils.quat_to_mat(q_inv, p_inv)

    # -------------------------
    # shadow pass
    # -------------------------
    if show_mesh:
        viewer.begin_shadow()
        viewer.draw(character_noinv, a_noinv, animation_noinv.bones)
        viewer.draw(character_inv, a_inv, animation_inv.bones)
        viewer.end_shadow()

    # -------------------------
    # display pass
    # -------------------------
    viewer.begin_display()
    viewer.draw_ground()

    if show_mesh:
        viewer.draw(character_noinv, a_noinv, animation_noinv.bones)
        viewer.draw(character_inv, a_inv, animation_inv.bones)

    viewer.end_display()

    # -------------------------
    # 可选：骨架线
    # -------------------------
    viewer.disable(depth_test=True)

    if show_skeleton:
        viewer.draw_axis(character_noinv.world_skeleton_xforms(a_noinv, animation_noinv.bones), 4)
        viewer.draw_lines(character_noinv.world_skeleton_lines(a_noinv, animation_noinv.bones),
                          color=np.array([0.7, 0.2, 0.2], dtype=np.float32))

        viewer.draw_axis(character_inv.world_skeleton_xforms(a_inv, animation_inv.bones), 4)
        viewer.draw_lines(character_inv.world_skeleton_lines(a_inv, animation_inv.bones),
                          color=np.array([0.2, 0.7, 0.2], dtype=np.float32))

    # -------------------------
    # 轨迹线：只画到当前 frame
    # -------------------------
    upto = frame + 1

    # 左边 no inverse
    if show_root_traj:
        pts = root_noinv_world[:upto:traj_step].copy()
        pts[:, 0] -= offset
        pts[:, 1] += 1.0   # 稍微抬高一点，避免和地面 z-fighting
        viewer.draw_lines(
            polyline_to_segments(pts),
            color=np.array([1.0, 0.4, 0.1], dtype=np.float32)   # 橙色 root
        )

    if show_hips_traj:
        pts = hips_noinv_world[:upto:traj_step].copy()
        pts[:, 0] -= offset
        viewer.draw_lines(
            polyline_to_segments(pts),
            color=np.array([1.0, 0.1, 0.7], dtype=np.float32)   # 洋红 hips
        )

    # 右边 inverse
    if show_root_traj:
        pts = root_inv_world[:upto:traj_step].copy()
        pts[:, 0] += offset
        pts[:, 1] += 1.0
        viewer.draw_lines(
            polyline_to_segments(pts),
            color=np.array([0.1, 0.9, 0.2], dtype=np.float32)   # 绿色 root
        )

    if show_hips_traj:
        pts = hips_inv_world[:upto:traj_step].copy()
        pts[:, 0] += offset
        viewer.draw_lines(
            polyline_to_segments(pts),
            color=np.array([0.1, 0.7, 1.0], dtype=np.float32)   # 青蓝 hips
        )

    viewer.execute_commands()


interact(
    render_project_root_compare_with_traj,
    frame=lab.Timeline(max=animation_inv.quats.shape[0] - 1),
    offset=(20, 120, 5),
    traj_step=(1, 20, 1),
    show_mesh=True,
    show_skeleton=False,
    show_root_traj=True,
    show_hips_traj=True
)

viewer

interactive(children=(Timeline(value=0, children=(Play(value=0, description='play', interval=33, layout=Layout…

Viewer(camera_far=2800.0, camera_near=20.0, camera_pitch=-16.399999999999952, camera_pos=[-4.572947051503212, …